In [0]:

dbutils.widgets.text("storage_acount", ".", "Adls")
storage_acount = dbutils.widgets.get("storage_acount")
dbutils.widgets.text("container", ".", "container")
container = dbutils.widgets.get("container")

In [0]:
abfs = f"abfss://{container}@{storage_acount}.dfs.core.windows.net/"

In [0]:
server_name = dbutils.secrets.get(scope="accessScope", key="server-db-name")
database_name = dbutils.secrets.get(scope="accessScope", key="database-name")
username = dbutils.secrets.get(scope="accessScope", key="sql-username")
password = dbutils.secrets.get(scope="accessScope", key="sql-password")

In [0]:
jdbcHostname = server_name
jdbcDatabase = database_name
jdbcPort = 1433
jdbcUsername = username
jdbcPassword = password

In [0]:
jdbcUrl = f"jdbc:sqlserver://{jdbcHostname}:{jdbcPort};database={jdbcDatabase};encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"

In [0]:
df_customers = (spark.read
    .format("jdbc")
    .option("url", jdbcUrl)
    .option("dbtable", "dbo.Customers")
    .option("user", jdbcUsername)
    .option("password", jdbcPassword)
    .load()
)
df_customers.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("catalog_dev.bronze.customers")

In [0]:
df_loans = (spark.read
    .format("jdbc")
    .option("url", jdbcUrl)
    .option("dbtable", "dbo.Loans")
    .option("user", jdbcUsername)
    .option("password", jdbcPassword)
    .load()
)
df_loans.write.mode("overwrite").saveAsTable("catalog_dev.bronze.loans")



In [0]:

# Payments
df_payments = (spark.read
    .format("jdbc")
    .option("url", jdbcUrl)
    .option("dbtable", "dbo.Payments")
    .option("user", jdbcUsername)
    .option("password", jdbcPassword)
    .load()
)
df_payments.write.mode("overwrite").saveAsTable("catalog_dev.bronze.payments")